In [1]:
import sys, time, gc
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, joblib

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.features import to_matrix, ClippedScaler
from src.evaluation import evaluate
from src.pipeline import DATA, RESULTS, score_split

SEED, L = 11, 16
d = joblib.load(RESULTS / "scaler_v2.joblib")
cols, scaler = d["cols"], d["scaler"]
dim = len(cols)

def make_ae():
    return nn.Sequential(
        nn.Linear(dim, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 12), nn.ReLU(),
        nn.Linear(12, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, dim))

ae_path = RESULTS / "m3v2_ae_seed11.pt"
if not ae_path.exists():
    torch.manual_seed(SEED); np.random.seed(SEED)
    tr = pd.read_parquet(DATA / "train.parquet", columns=cols).sample(1_000_000, random_state=SEED)
    x_train = scaler.transform(to_matrix(tr, cols)).astype(np.float32)
    va = pd.read_parquet(DATA / "val.parquet", columns=cols).sample(100_000, random_state=SEED)
    x_val = scaler.transform(to_matrix(va, cols)).astype(np.float32)
    del tr, va; gc.collect()

    model = make_ae()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    Xtr, Xva = torch.from_numpy(x_train), torch.from_numpy(x_val)
    best, best_state, bad = float("inf"), None, 0
    for epoch in range(30):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 1024):
            xb = Xtr[perm[i:i + 1024]]
            opt.zero_grad(); loss_fn(model(xb), xb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v = loss_fn(model(Xva), Xva).item()
        print(f"epoch {epoch + 1:2d}  val loss {v:.5f}")
        if v < best - 1e-4:
            best, bad = v, 0
            best_state = {k: t.clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= 3:
                break
    torch.save(best_state, ae_path)
    del x_train, x_val, Xtr, Xva, model; gc.collect()

ae = make_ae(); ae.load_state_dict(torch.load(ae_path)); ae.eval()
print("M3 v2 ready")

epoch  1  val loss 0.02526
epoch  2  val loss 0.01401
epoch  3  val loss 0.01039
epoch  4  val loss 0.00846
epoch  5  val loss 0.00691
epoch  6  val loss 0.00635
epoch  7  val loss 0.00531
epoch  8  val loss 0.00460
epoch  9  val loss 0.00408
epoch 10  val loss 0.00421
epoch 11  val loss 0.00375
epoch 12  val loss 0.00376
epoch 13  val loss 0.00344
epoch 14  val loss 0.00325
epoch 15  val loss 0.00332
epoch 16  val loss 0.00306
epoch 17  val loss 0.00298
epoch 18  val loss 0.00296
epoch 19  val loss 0.00283
epoch 20  val loss 0.00263
epoch 21  val loss 0.00263
epoch 22  val loss 0.00257
epoch 23  val loss 0.00279
M3 v2 ready


In [2]:
def ae_score(x):
    out = []
    with torch.no_grad():
        for i in range(0, len(x), 100_000):
            xb = torch.from_numpy(x[i:i + 100_000])
            out.append(((ae(xb) - xb) ** 2).mean(dim=1).numpy())
    return np.concatenate(out)

val_flow = score_split("val", cols, scaler, ae_score)      # ToN-IoT benign validation
unsw_flow = score_split("unsw", cols, scaler, ae_score)    # every UNSW flow
print(len(val_flow), len(unsw_flow))

539110 2390275


In [3]:
from src.windows import assign_windows, window_table, select_windows, build_tensors
from src.tcn import TCNMAE, window_scores

meta = pd.read_parquet(DATA / "unsw.parquet", columns=["row_id", "IPV4_SRC_ADDR", "Attack"])
m, host_names, atk_names = assign_windows(meta, L)
del meta; gc.collect()
w = window_table(m, host_names, atk_names)
sel = select_windows(w, max_benign=50_000, cap_per_type=None, seed=SEED)   # all attack windows kept
X, mask = build_tensors("unsw", m, sel, cols, scaler, L)
ws = w.loc[sel]
fmap_u = m[m["win_id"].isin(sel)][["row_id", "win_id", "pos"]]
print("UNSW windows:", X.shape, "| share full:", round(float((ws["n_flows"] == L).mean()), 3))
print(ws["attack_type"].value_counts().to_dict())

m4 = TCNMAE(dim); m4.load_state_dict(torch.load(RESULTS / "m4_tcn_seed11.pt")); m4.eval()

zv = np.load(DATA / "windows" / "val.npz", allow_pickle=True)
val_m4 = pd.DataFrame({"id": zv["win_ids"], "score": window_scores(m4, zv["X"], zv["mask"]),
                       "label": zv["label"].astype(int), "attack_type": zv["attack_type"].astype(str),
                       "split": "val"})
unsw_m4 = pd.DataFrame({"id": sel, "score": window_scores(m4, X, mask),
                        "label": ws["label"].to_numpy().astype(int),
                        "attack_type": ws["attack_type"].astype(str).to_numpy(), "split": "unsw"})

UNSW windows: (56957, 16, 28) | share full: 1.0
{'Benign': 50000, 'Exploits': 2324, 'Fuzzers': 1581, 'Generic': 1175, 'Reconnaissance': 985, 'DoS': 447, 'Analysis': 165, 'Backdoor': 152, 'Shellcode': 115, 'Worms': 13}


In [4]:
from sklearn.metrics import roc_auc_score

def window_max(flow_df, fmap, ids, label, atype, split):
    j = fmap.merge(flow_df[["id", "score"]], left_on="row_id", right_on="id")
    out = pd.DataFrame({"id": ids, "label": label, "attack_type": atype, "split": split})
    out["score"] = out["id"].map(j.groupby("win_id")["score"].max())
    return out.dropna(subset=["score"])

val_w3 = window_max(val_flow, pd.read_parquet(DATA / "windows" / "val_flowmap.parquet"),
                    zv["win_ids"], zv["label"].astype(int), zv["attack_type"].astype(str), "val")
unsw_w3 = window_max(unsw_flow, fmap_u, sel, ws["label"].to_numpy().astype(int),
                     ws["attack_type"].astype(str).to_numpy(), "unsw")

runs = [("M3 v2 (flow level)", unsw_flow, val_flow),
        ("M3 v2 max (window level)", unsw_w3, val_w3),
        ("M4 TCN (window level)", unsw_m4, val_m4)]

rows = []
for name, t, ref in runs:
    r, o = evaluate(t, val_df=ref), evaluate(t)
    rows.append({"model": name, "auroc": r["auroc"], "tpr@1%": r["tpr@fpr0.01"],
                 "real_fpr@1%": r["real_fpr@0.01"], "macro_tpr@5%": r["macro_tpr@fpr0.05"],
                 "real_fpr@5%": r["real_fpr@0.05"], "oracle_macro_tpr@5%": o["macro_tpr@fpr0.05"]})
cross = pd.DataFrame(rows).round(3)
cross.to_csv(RESULTS / "cross_dataset_unsw.csv", index=False)
print(cross.to_string(index=False))

def per_attack_auroc(x):
    b = x[x["label"] == 0]["score"].to_numpy()
    return {t: roc_auc_score(np.r_[np.zeros(len(b)), np.ones(len(g))], np.r_[b, g["score"].to_numpy()])
            for t, g in x[x["label"] == 1].groupby("attack_type")}
print(pd.DataFrame({n: per_attack_auroc(t) for n, t, _ in runs}).round(3))

                   model  auroc  tpr@1%  real_fpr@1%  macro_tpr@5%  real_fpr@5%  oracle_macro_tpr@5%
      M3 v2 (flow level)  0.359   0.580        0.769         1.000        1.000                0.015
M3 v2 max (window level)  0.236   0.328        0.821         0.973        1.000                0.139
   M4 TCN (window level)  0.392   0.494        0.699         0.940        0.859                0.048
                M3 v2 (flow level)  M3 v2 max (window level)  \
Analysis                     0.541                     0.055   
Backdoor                     0.584                     0.109   
DoS                          0.445                     0.228   
Exploits                     0.348                     0.315   
Fuzzers                      0.413                     0.216   
Generic                      0.291                     0.152   
Reconnaissance               0.281                     0.240   
Shellcode                    0.263                     0.220   
Worms               